# Check Metrics
Consulta el endpoint con audios de muestra y mide **errores** (requests fallidas) y **latencia** (p95). `force_fail=true` fuerza `healthy=false` (para probar el rollback).
Salida (task values): `healthy`, `error_rate`, `p95_latency_ms`, `wer`.

In [0]:
%pip install -q jiwer soundfile databricks-sdk

In [0]:
%run ../notebooks/utils/00_config

In [0]:
import json, time, statistics
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import serving

ENDPOINT_NAME = dbutils.widgets.get("endpoint_name")
FORCE_FAIL = dbutils.widgets.get("force_fail", "false").lower() == "true"
LAT_THRESHOLD_MS = float(dbutils.widgets.get("latency_threshold_ms", "5000"))
N = int(dbutils.widgets.get("n_samples", "5"))

w = WorkspaceClient()
sample = [p for p in audio_files() if read_ref(p)][:N] or audio_files()[:N]

latencies, errors, wers = [], 0, []
for p in sample:
    t0 = time.time()
    try:
        resp = w.serving_endpoints.query(
            name=ENDPOINT_NAME,
            dataframe_split=serving.DataframeSplitInput(columns=["audio_path"], data=[[p]]))
        latencies.append((time.time() - t0) * 1000)
        preds = getattr(resp, "predictions", None)
        hyp = preds[0] if preds else None
        if isinstance(hyp, str):
            try: hyp = json.loads(hyp)["text"]
            except Exception: pass
        ref = read_ref(p)
        if ref and isinstance(hyp, str):
            wers.append(jiwer.wer(NORM(ref), NORM(hyp)))
    except Exception as e:
        errors += 1
        print("query error", p, e)

error_rate = errors / max(len(sample), 1)
if len(latencies) >= 2:
    p95 = float(statistics.quantiles(latencies, n=100)[94])
elif latencies:
    p95 = float(latencies[0])
else:
    p95 = 1e9
wer = float(statistics.mean(wers)) if wers else float("nan")

healthy = (not FORCE_FAIL) and (error_rate == 0) and (p95 <= LAT_THRESHOLD_MS)

print(f"force_fail={FORCE_FAIL} errors={errors}/{len(sample)} error_rate={error_rate:.2f} p95_ms={p95:.0f} wer={wer} -> healthy={healthy}")

dbutils.jobs.taskValues.set("healthy", bool(healthy))
dbutils.jobs.taskValues.set("error_rate", float(error_rate))
dbutils.jobs.taskValues.set("p95_latency_ms", float(p95))
dbutils.jobs.taskValues.set("wer", float(wer) if wer == wer else -1.0)